# EDS vs Queensland validated clearing comparison — DLL Proportion Rule

This notebook compares Queensland validated clearing polygons against your DLL classified raster outputs.


In [44]:

import re
import os
import tempfile
from pathlib import Path

import boto3
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
from shapely.geometry import mapping

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)


In [45]:
# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------

MY_BUCKET = "dcceew-eds-data"
MY_PREFIX = "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/"

QLD_BUCKET = "dcceew-rs-data"
QLD_PREFIX = "dcceew_202602_run/"
QLD_TEXT_FILTER = "completed_checked"

# Evaluate the DLL classified raster
MY_PRODUCT = "dll"

# Australian Albers for combined Australia-wide outputs
COMMON_CRS = "EPSG:3577"

# DLL classes
DLL_NO_CHANGE_CLASS = 10
DLL_CLEARING_CLASSES = {34, 35, 36, 37, 38, 39}
DLL_CLASSES_OF_INTEREST = [10, 34, 35, 36, 37, 38, 39]

# Prediction rule:
# "majority" = polygon predicted cleared when the majority raster class is one of DLL_CLEARING_CLASSES
# "proportion" = polygon predicted cleared when enough pixels fall in DLL_CLEARING_CLASSES
PREDICTION_METHOD = "proportion"

# Only used if PREDICTION_METHOD = "proportion"
# Example: 0.5 means at least 50% of pixels in the polygon must be in 34-39.
MIN_PROP_TARGET_CLASS = 0.5


In [46]:

def list_s3_keys(bucket, prefix=""):
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    rows = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            rows.append({
                "bucket": bucket,
                "key": obj["Key"],
                "size": obj["Size"],
            })

    return pd.DataFrame(rows)


def find_keys_containing(bucket, prefix="", text="completed_checked", max_show=50):
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")

    matches = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if text.lower() in key.lower():
                matches.append(key)

    print(f"Found {len(matches)} keys containing '{text}' under '{prefix}'\n")

    for k in matches[:max_show]:
        print(k)

    if len(matches) > max_show:
        print(f"\n... and {len(matches) - max_show} more")

    return matches


def extract_tile(key):
    m = re.search(r"(p\d{3}r\d{3})", key)
    return m.group(1) if m else None


def extract_date_group(key):
    m = re.search(r"(d\d{16})", key)
    if m:
        return m.group(1)

    m = re.search(r"(?<!\d)(\d{8})(?!\d)", key)
    if m:
        return m.group(1)

    return None


def extract_start_date(date_group):
    if pd.isna(date_group):
        return None

    value = str(date_group)

    m = re.fullmatch(r"d(\d{8})(\d{8})", value)
    if m:
        return m.group(1)

    m = re.fullmatch(r"(\d{8})", value)
    if m:
        return m.group(1)

    m = re.search(r"(?<!\d)(\d{8})(?!\d)", value)
    if m:
        return m.group(1)

    return None


def classify_my_output(key):
    kl = key.lower()
    if not kl.endswith(".tif"):
        return "other"
    if "vi-ndvi_dljmz" in kl:
        return "vi_ndvi_dljmz"
    if "vi-ndvi_dllmz" in kl:
        return "vi_ndvi_dllmz"
    if "_dlj_" in kl:
        return "dlj"
    if "_dll_" in kl:
        return "dll"
    return "raster_other"


def summarise_df(df, name, tile_col="tile", date_col="date_group"):
    print(f"\n===== {name} =====")
    print("Rows:", len(df))

    if "product" in df.columns:
        print("\nProducts:")
        print(df["product"].value_counts(dropna=False))

    if tile_col in df.columns:
        print("\nTiles:")
        print(df[tile_col].value_counts(dropna=False).head(20))

    if date_col in df.columns:
        print("\nDate groups:")
        print(df[date_col].value_counts(dropna=False).head(20))


## 1. Build the Queensland validated shapefile inventory

This uses the known `completed_checked` filter under `dcceew_202602_run/`.


In [47]:

completed_checked_keys = find_keys_containing(
    bucket=QLD_BUCKET,
    prefix=QLD_PREFIX,
    text=QLD_TEXT_FILTER
)

completed_checked_shps = [
    k for k in completed_checked_keys
    if k.lower().endswith(".shp")
]

qld_df = pd.DataFrame({"key_qld": completed_checked_shps})
qld_df["tile"] = qld_df["key_qld"].apply(extract_tile)
qld_df["date_group_qld"] = qld_df["key_qld"].apply(extract_date_group)
qld_df["start_date"] = qld_df["date_group_qld"].apply(extract_start_date)

summarise_df(qld_df, "Queensland validated shapefiles", tile_col="tile", date_col="date_group_qld")
qld_df.head(20)


Found 175 keys containing 'completed_checked' under 'dcceew_202602_run/'

dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.cpg
dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.dbf
dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.prj
dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.sbn
dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.sbx
dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shp
dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shx
dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed/lzolre_p090r077_d2

,key_qld,tile,date_group_qld,start_date
0,dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shp,p089r078,d2025070120260117,20250701
1,dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed/lzolre_p090r077_d2025081720260201_dlwm6_completed_checked.shp,p090r077,d2025081720260201,20250817
2,dcceew_202602_run/lzolre_p090r079_d2025100420260201_dlwm6/lzolre_p090r079_d2025100420260201_dlwm6_completed_checked.shp,p090r079,d2025100420260201,20251004
3,dcceew_202602_run/lzolre_p090r086_d2025110520260201_dlwm5/lzolre_p090r086_d2025110520260201_dlwm5_completed_checked.shp,p090r086,d2025110520260201,20251105
4,dcceew_202602_run/lzolre_p091r076_d2025081620260131_dlwm6/lzolre_p091r076_d2025081620260131_dlwm6_completed_checked.shp,p091r076,d2025081620260131,20250816
5,dcceew_202602_run/lzolre_p091r077_d2025091720260131_dlwm6/lzolre_p091r077_d2025091720260131_dlwm6_completed_checked.shp,p091r077,d2025091720260131,20250917
6,dcceew_202602_run/lzolre_p092r076_d2025100220260122_dlwm5/lzolre_p092r076_d2025100220260122_dlwm5_completed_checked.shp,p092r076,d2025100220260122,20251002
7,dcceew_202602_run/lzolre_p092r077_d2025112820260123_dlwm5/lzolre_p092r077_d2025112820260123_dlwm5_completed_checked.shp,p092r077,d2025112820260123,20251128
8,dcceew_202602_run/lzolre_p092r088_d2025073120260123_dlwm5/lzolre_p092r088_d2025073120260123_dlwm5_completed_checked.shp,p092r088,d2025073120260123,20250731
9,dcceew_202602_run/lzolre_p093r075_d2025101820260122_dlwm5/lzolre_p093r075_d2025101820260122_dlwm5_completed_checked.shp,p093r075,d2025101820260122,20251018


## 2. Build your EDS output inventory

This looks only under your `tiles/` prefix and keeps the `.tif` rasters under `/outputs/`.


In [48]:

df_my = list_s3_keys(MY_BUCKET, MY_PREFIX)

my_outputs = df_my[
    df_my["key"].str.contains("/outputs/", case=False, na=False)
].copy()

my_outputs = my_outputs[
    my_outputs["key"].str.lower().str.endswith(".tif")
].copy()

my_outputs["tile"] = my_outputs["key"].apply(extract_tile)
my_outputs["date_group_my"] = my_outputs["key"].apply(extract_date_group)
my_outputs["start_date"] = my_outputs["date_group_my"].apply(extract_start_date)
my_outputs["product"] = my_outputs["key"].apply(classify_my_output)

summarise_df(my_outputs, "My EDS outputs", tile_col="tile", date_col="date_group_my")
my_outputs.head(20)



===== My EDS outputs =====
Rows: 52

Products:
product
raster_other    26
dlj             13
dll             13
Name: count, dtype: int64

Tiles:
tile
p089r078    4
p089r079    4
p089r080    4
p089r081    4
p089r082    4
p090r077    4
p090r079    4
p090r086    4
p091r076    4
p091r077    4
p092r076    4
p092r077    4
p092r088    4
Name: count, dtype: int64

Date groups:
date_group_my
d2025060720260125    8
d2025082620260125    8
d2025070120260117    4
d2025081720260201    4
d2025100420260201    4
d2025110520260201    4
d2025081620260131    4
d2025091720260131    4
d2025100220260122    4
d2025112820260123    4
d2025073120260123    4
Name: count, dtype: int64


,bucket,key,size,tile,date_group_my,start_date,product
392,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/l9olre_p089r078_d2025070120260117_dlj_e32756.tif,21916010,p089r078,d2025070120260117,20250701,dlj
393,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/l9olre_p089r078_d2025070120260117_dll_e32756.tif,1963425,p089r078,d2025070120260117,20250701,dll
394,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/masks/l9olre_p089r078_d2025070120260117_dlj-dlj-clear-ge80_e32756.tif,1633949,p089r078,d2025070120260117,20250701,raster_other
395,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/masks/l9olre_p089r078_d2025070120260117_dlj-dlj-strong-ge60_e32756.tif,1655037,p089r078,d2025070120260117,20250701,raster_other
840,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/outputs/p089r079_d2025060720260125/l8olre_p089r079_d2025060720260125_dlj_e32756.tif,63579962,p089r079,d2025060720260125,20250607,dlj
841,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/outputs/p089r079_d2025060720260125/l8olre_p089r079_d2025060720260125_dll_e32756.tif,7387353,p089r079,d2025060720260125,20250607,dll
842,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/outputs/p089r079_d2025060720260125/masks/l8olre_p089r079_d2025060720260125_dlj-dlj-clear-ge80_e32756.tif,2135565,p089r079,d2025060720260125,20250607,raster_other
843,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/outputs/p089r079_d2025060720260125/masks/l8olre_p089r079_d2025060720260125_dlj-dlj-strong-ge60_e32756.tif,2425185,p089r079,d2025060720260125,20250607,raster_other
1252,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r080/outputs/p089r080_d2025060720260125/l8olre_p089r080_d2025060720260125_dlj_e32756.tif,106859574,p089r080,d2025060720260125,20250607,dlj
1253,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r080/outputs/p089r080_d2025060720260125/l8olre_p089r080_d2025060720260125_dll_e32756.tif,13164315,p089r080,d2025060720260125,20250607,dll


### 2.b look at data

In [49]:
import pandas as pd
pd.set_option("display.max_colwidth", 200)

qld_view = qld_df[[
    "tile",
    "date_group_qld",
    "start_date",
    "key_qld"
]].copy()

print("Queensland shapefiles:")
display(qld_view.sort_values(["tile", "start_date"]).head(50))

Queensland shapefiles:


,tile,date_group_qld,start_date,key_qld
0,p089r078,d2025070120260117,20250701,dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shp
1,p090r077,d2025081720260201,20250817,dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed/lzolre_p090r077_d2025081720260201_dlwm6_completed_checked.shp
2,p090r079,d2025100420260201,20251004,dcceew_202602_run/lzolre_p090r079_d2025100420260201_dlwm6/lzolre_p090r079_d2025100420260201_dlwm6_completed_checked.shp
3,p090r086,d2025110520260201,20251105,dcceew_202602_run/lzolre_p090r086_d2025110520260201_dlwm5/lzolre_p090r086_d2025110520260201_dlwm5_completed_checked.shp
4,p091r076,d2025081620260131,20250816,dcceew_202602_run/lzolre_p091r076_d2025081620260131_dlwm6/lzolre_p091r076_d2025081620260131_dlwm6_completed_checked.shp
5,p091r077,d2025091720260131,20250917,dcceew_202602_run/lzolre_p091r077_d2025091720260131_dlwm6/lzolre_p091r077_d2025091720260131_dlwm6_completed_checked.shp
6,p092r076,d2025100220260122,20251002,dcceew_202602_run/lzolre_p092r076_d2025100220260122_dlwm5/lzolre_p092r076_d2025100220260122_dlwm5_completed_checked.shp
7,p092r077,d2025112820260123,20251128,dcceew_202602_run/lzolre_p092r077_d2025112820260123_dlwm5/lzolre_p092r077_d2025112820260123_dlwm5_completed_checked.shp
8,p092r088,d2025073120260123,20250731,dcceew_202602_run/lzolre_p092r088_d2025073120260123_dlwm5/lzolre_p092r088_d2025073120260123_dlwm5_completed_checked.shp
9,p093r075,d2025101820260122,20251018,dcceew_202602_run/lzolre_p093r075_d2025101820260122_dlwm5/lzolre_p093r075_d2025101820260122_dlwm5_completed_checked.shp


In [50]:
my_view = my_outputs[[
    "tile",
    "product",
    "date_group_my",
    "start_date",
    "key"
]].copy()

print("My EDS rasters:")
display(my_view.sort_values(["tile", "start_date", "product"]).head(100))

My EDS rasters:


,tile,product,date_group_my,start_date,key
392,p089r078,dlj,d2025070120260117,20250701,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/l9olre_p089r078_d2025070120260117_dlj_e32756.tif
393,p089r078,dll,d2025070120260117,20250701,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/l9olre_p089r078_d2025070120260117_dll_e32756.tif
394,p089r078,raster_other,d2025070120260117,20250701,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/masks/l9olre_p089r078_d2025070120260117_dlj-dlj-clear-ge80_e32756.tif
395,p089r078,raster_other,d2025070120260117,20250701,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/masks/l9olre_p089r078_d2025070120260117_dlj-dlj-strong-ge60_e32756.tif
840,p089r079,dlj,d2025060720260125,20250607,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/outputs/p089r079_d2025060720260125/l8olre_p089r079_d2025060720260125_dlj_e32756.tif
841,p089r079,dll,d2025060720260125,20250607,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/outputs/p089r079_d2025060720260125/l8olre_p089r079_d2025060720260125_dll_e32756.tif
842,p089r079,raster_other,d2025060720260125,20250607,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/outputs/p089r079_d2025060720260125/masks/l8olre_p089r079_d2025060720260125_dlj-dlj-clear-ge80_e32756.tif
843,p089r079,raster_other,d2025060720260125,20250607,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/outputs/p089r079_d2025060720260125/masks/l8olre_p089r079_d2025060720260125_dlj-dlj-strong-ge60_e32756.tif
1252,p089r080,dlj,d2025060720260125,20250607,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r080/outputs/p089r080_d2025060720260125/l8olre_p089r080_d2025060720260125_dlj_e32756.tif
1253,p089r080,dll,d2025060720260125,20250607,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r080/outputs/p089r080_d2025060720260125/l8olre_p089r080_d2025060720260125_dll_e32756.tif


## 3. Match Queensland shapefiles to your EDS outputs

Start by matching on **tile + start_date**. If that returns too few matches, inspect the keys and adjust the rule.


In [51]:

my_eval_rasters = my_outputs[my_outputs["product"] == MY_PRODUCT].copy()

matches = qld_df.merge(
    my_eval_rasters,
    on=["tile", "start_date"],
    suffixes=("_qld", "_my")
)

print(f"Matched {len(matches)} Queensland shapefiles to '{MY_PRODUCT}' rasters")
matches[["tile", "start_date", "date_group_qld", "date_group_my", "key_qld", "key"]].head(20)


Matched 9 Queensland shapefiles to 'dll' rasters


,tile,start_date,date_group_qld,date_group_my,key_qld,key
0,p089r078,20250701,d2025070120260117,d2025070120260117,dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/l9olre_p089r078_d2025070120260117_dll_e32756.tif
1,p090r077,20250817,d2025081720260201,d2025081720260201,dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed/lzolre_p090r077_d2025081720260201_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r077/outputs/p090r077_d2025081720260201/l8olre_p090r077_d2025081720260201_dll_e32756.tif
2,p090r079,20251004,d2025100420260201,d2025100420260201,dcceew_202602_run/lzolre_p090r079_d2025100420260201_dlwm6/lzolre_p090r079_d2025100420260201_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r079/outputs/p090r079_d2025100420260201/l8olre_p090r079_d2025100420260201_dll_e32756.tif
3,p090r086,20251105,d2025110520260201,d2025110520260201,dcceew_202602_run/lzolre_p090r086_d2025110520260201_dlwm5/lzolre_p090r086_d2025110520260201_dlwm5_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r086/outputs/p090r086_d2025110520260201/l8olre_p090r086_d2025110520260201_dll_e32755.tif
4,p091r076,20250816,d2025081620260131,d2025081620260131,dcceew_202602_run/lzolre_p091r076_d2025081620260131_dlwm6/lzolre_p091r076_d2025081620260131_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p091r076/outputs/p091r076_d2025081620260131/l9olre_p091r076_d2025081620260131_dll_e32756.tif
5,p091r077,20250917,d2025091720260131,d2025091720260131,dcceew_202602_run/lzolre_p091r077_d2025091720260131_dlwm6/lzolre_p091r077_d2025091720260131_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p091r077/outputs/p091r077_d2025091720260131/l9olre_p091r077_d2025091720260131_dll_e32756.tif
6,p092r076,20251002,d2025100220260122,d2025100220260122,dcceew_202602_run/lzolre_p092r076_d2025100220260122_dlwm5/lzolre_p092r076_d2025100220260122_dlwm5_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p092r076/outputs/p092r076_d2025100220260122/l9olre_p092r076_d2025100220260122_dll_e32755.tif
7,p092r077,20251128,d2025112820260123,d2025112820260123,dcceew_202602_run/lzolre_p092r077_d2025112820260123_dlwm5/lzolre_p092r077_d2025112820260123_dlwm5_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p092r077/outputs/p092r077_d2025112820260123/l9olre_p092r077_d2025112820260123_dll_e32755.tif
8,p092r088,20250731,d2025073120260123,d2025073120260123,dcceew_202602_run/lzolre_p092r088_d2025073120260123_dlwm5/lzolre_p092r088_d2025073120260123_dlwm5_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p092r088/outputs/p092r088_d2025073120260123/l9olre_p092r088_d2025073120260123_dll_e32755.tif


In [52]:

# Optional: inspect one tile if you need to diagnose matching
tile_to_inspect = "p089r078"

print("Queensland shapefiles for tile:")
display(qld_df[qld_df["tile"] == tile_to_inspect][["tile", "date_group_qld", "start_date", "key_qld"]].head(20))

print("\nMy outputs for tile:")
display(my_eval_rasters[my_eval_rasters["tile"] == tile_to_inspect][["tile", "date_group_my", "start_date", "product", "key"]].head(20))


Queensland shapefiles for tile:


,tile,date_group_qld,start_date,key_qld
0,p089r078,d2025070120260117,20250701,dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shp



My outputs for tile:


,tile,date_group_my,start_date,product,key
393,p089r078,d2025070120260117,20250701,dll,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/l9olre_p089r078_d2025070120260117_dll_e32756.tif


## 4. Download helpers

A shapefile needs the `.shp` plus sidecar files like `.dbf`, `.shx`, and `.prj`.


In [53]:

def download_s3_file(bucket, key, local_path):
    s3 = boto3.client("s3")
    local_path.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(bucket, key, str(local_path))


def download_shapefile_bundle(bucket, shp_key, out_dir):
    base = os.path.splitext(shp_key)[0]
    exts = [".shp", ".dbf", ".shx", ".prj", ".cpg"]

    downloaded = []
    for ext in exts:
        key = base + ext
        local_path = Path(out_dir) / Path(key).name
        try:
            download_s3_file(bucket, key, local_path)
            downloaded.append(local_path)
        except Exception:
            pass

    shp_local = Path(out_dir) / (Path(base).name + ".shp")
    return shp_local, downloaded


def download_raster(bucket, key, out_dir):
    local_path = Path(out_dir) / Path(key).name
    download_s3_file(bucket, key, local_path)
    return local_path


## 5. Compare one matched pair

This reads one Queensland shapefile, filters `IsClearing == 'Y'`, samples your raster inside those polygons, and calculates whether your EDS predicted clearing.

You may need to adjust:
- `MY_PRODUCT`
- `RASTER_THRESHOLD`
- `MIN_PROP_ABOVE_THRESHOLD`
- the exact validation field name if it is not `IsClearing`


In [54]:
def polygon_raster_stats(raster_path, gdf, clearing_classes=None, classes_of_interest=None):
    """
    For each validation polygon, calculate DLL class composition.
    """
    results = []

    if clearing_classes is None:
        clearing_classes = set()

    if classes_of_interest is None:
        classes_of_interest = []

    with rasterio.open(raster_path) as src:
        nodata = src.nodata
        gdf = gdf.to_crs(src.crs)

        for idx, row in gdf.iterrows():
            geom = [mapping(row.geometry)]

            try:
                out_image, _ = mask(src, geom, crop=True)
                data = out_image[0]

                if nodata is not None:
                    data = data[data != nodata]

                data = data[np.isfinite(data)]

                res = {
                    "index": idx,
                    "pixel_count": 0,
                    "min": np.nan,
                    "mean": np.nan,
                    "max": np.nan,
                    "majority_value": np.nan,
                    "unique_values": "",
                    "prop_clearing_classes": np.nan,
                }

                for cls in classes_of_interest:
                    res[f"class_{cls}_count"] = 0
                    res[f"class_{cls}_prop"] = np.nan
                    res[f"class_{cls}_pct"] = np.nan

                if data.size == 0:
                    results.append(res)
                    continue

                data_int = data.astype(int)

                unique_vals, counts = np.unique(data_int, return_counts=True)
                count_map = dict(zip(unique_vals.tolist(), counts.tolist()))

                total_pixels = int(data_int.size)
                majority_value = int(unique_vals[np.argmax(counts)])
                unique_values_str = ",".join(map(str, unique_vals.tolist()))
                prop_clearing = float(np.mean(np.isin(data_int, list(clearing_classes))))

                res.update({
                    "pixel_count": total_pixels,
                    "min": int(np.min(data_int)),
                    "mean": float(np.mean(data_int)),
                    "max": int(np.max(data_int)),
                    "majority_value": majority_value,
                    "unique_values": unique_values_str,
                    "prop_clearing_classes": prop_clearing,
                })

                for cls in classes_of_interest:
                    cls_count = int(count_map.get(cls, 0))
                    cls_prop = cls_count / total_pixels if total_pixels > 0 else np.nan
                    res[f"class_{cls}_count"] = cls_count
                    res[f"class_{cls}_prop"] = cls_prop
                    res[f"class_{cls}_pct"] = cls_prop * 100 if pd.notna(cls_prop) else np.nan

                results.append(res)

            except Exception:
                res = {
                    "index": idx,
                    "pixel_count": 0,
                    "min": np.nan,
                    "mean": np.nan,
                    "max": np.nan,
                    "majority_value": np.nan,
                    "unique_values": "",
                    "prop_clearing_classes": np.nan,
                }

                for cls in classes_of_interest:
                    res[f"class_{cls}_count"] = 0
                    res[f"class_{cls}_prop"] = np.nan
                    res[f"class_{cls}_pct"] = np.nan

                results.append(res)

    return pd.DataFrame(results)


def find_clearing_field(gdf):
    col_map = {col.lower(): col for col in gdf.columns}

    for name in ["isclearing", "iscleared", "cleared", "clearing", "esastatus", "status"]:
        if name in col_map:
            return col_map[name]

    for lower_col, original_col in col_map.items():
        if any(token in lower_col for token in ["clear", "clearing", "esa"]):
            return original_col

    return None


def normalise_truth_value(value):
    if pd.isna(value):
        return None

    value = str(value).upper().strip()

    if value in {"Y", "YES", "TRUE", "1"}:
        return 1
    if value in {"N", "NO", "FALSE", "0"}:
        return 0

    return None


def evaluate_match(match_row):
    with tempfile.TemporaryDirectory() as tmpdir:
        shp_local, downloaded_files = download_shapefile_bundle(
            bucket=QLD_BUCKET,
            shp_key=match_row["key_qld"],
            out_dir=tmpdir
        )

        raster_local = download_raster(
            bucket=MY_BUCKET,
            key=match_row["key"],
            out_dir=tmpdir
        )

        qld_gdf = gpd.read_file(shp_local)
        clearing_field = find_clearing_field(qld_gdf)

        if clearing_field is None:
            raise ValueError(
                f"No clearing field found in shapefile. Columns were: {list(qld_gdf.columns)}"
            )

        qld_gdf["_clearing_value"] = (
            qld_gdf[clearing_field]
            .astype(str)
            .str.upper()
            .str.strip()
        )

        qld_gdf["truth"] = qld_gdf["_clearing_value"].apply(normalise_truth_value)
        qld_truth = qld_gdf[qld_gdf["truth"].notna()].copy()

        if qld_truth.empty:
            raise ValueError("No usable Y/N truth values found in shapefile.")

        stats_df = polygon_raster_stats(
            raster_local,
            qld_truth,
            clearing_classes=DLL_CLEARING_CLASSES,
            classes_of_interest=DLL_CLASSES_OF_INTEREST,
        )

        qld_eval = qld_truth.join(stats_df.set_index("index"))

        if PREDICTION_METHOD == "majority":
            qld_eval["predicted_cleared"] = qld_eval["majority_value"].isin(DLL_CLEARING_CLASSES)
        elif PREDICTION_METHOD == "proportion":
            qld_eval["predicted_cleared"] = (
                qld_eval["prop_clearing_classes"] >= MIN_PROP_TARGET_CLASS
            )
        else:
            raise ValueError(f"Unknown PREDICTION_METHOD: {PREDICTION_METHOD}")

        qld_eval["pred"] = qld_eval["predicted_cleared"].astype(int)

        valid_eval = qld_eval[qld_eval["pred"].notna() & qld_eval["truth"].notna()].copy()

        tp = int(((valid_eval["truth"] == 1) & (valid_eval["pred"] == 1)).sum())
        fn = int(((valid_eval["truth"] == 1) & (valid_eval["pred"] == 0)).sum())
        fp = int(((valid_eval["truth"] == 0) & (valid_eval["pred"] == 1)).sum())
        tn = int(((valid_eval["truth"] == 0) & (valid_eval["pred"] == 0)).sum())

        summary = {
            "tile": match_row["tile"],
            "start_date": match_row["start_date"],
            "date_group_qld": match_row.get("date_group_qld"),
            "date_group_my": match_row.get("date_group_my"),
            "key_qld": match_row.get("key_qld"),
            "key": match_row.get("key"),
            "product": match_row.get("product", MY_PRODUCT),
            "prediction_method": PREDICTION_METHOD,
            "DLL_CLEARING_CLASSES": str(sorted(DLL_CLEARING_CLASSES)),
            "MIN_PROP_TARGET_CLASS": MIN_PROP_TARGET_CLASS if PREDICTION_METHOD == "proportion" else np.nan,

            "total_truth_polygons": int(len(qld_truth)),
            "usable_polygons": int(len(valid_eval)),
            "validated_cleared_polygons": int((valid_eval["truth"] == 1).sum()),
            "validated_not_cleared_polygons": int((valid_eval["truth"] == 0).sum()),

            "tp": tp,
            "fn": fn,
            "fp": fp,
            "tn": tn,

            "detected_by_my_eds": tp,
            "missed_by_my_eds": fn,
        }

        summary["recall"] = tp / (tp + fn) if (tp + fn) else np.nan
        summary["precision"] = tp / (tp + fp) if (tp + fp) else np.nan
        summary["specificity"] = tn / (tn + fp) if (tn + fp) else np.nan
        summary["accuracy"] = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else np.nan
        summary["f1"] = (
            2 * summary["precision"] * summary["recall"] / (summary["precision"] + summary["recall"])
            if pd.notna(summary["precision"]) and pd.notna(summary["recall"]) and (summary["precision"] + summary["recall"]) > 0
            else np.nan
        )

        return summary, qld_eval, qld_gdf


In [55]:
# Run one matched pair first
match_index = 0

if len(matches) == 0:
    raise ValueError("No matches found. Inspect the tile/date logic above first.")

single_summary, single_eval, single_full_gdf = evaluate_match(matches.iloc[match_index])

pd.DataFrame([single_summary])

,tile,start_date,date_group_qld,date_group_my,key_qld,key,product,prediction_method,DLL_CLEARING_CLASSES,MIN_PROP_TARGET_CLASS,...,fn,fp,tn,detected_by_my_eds,missed_by_my_eds,recall,precision,specificity,accuracy,f1
0,p089r078,20250701,d2025070120260117,d2025070120260117,dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/l9olre_p089r078_d2025070120260117_dll_e32756.tif,dll,proportion,"[34, 35, 36, 37, 38, 39]",0.5,...,0,0,0,1,0,1.0,1.0,NaN,1.0,1.0


In [56]:

print("Total polygons in shapefile:", len(single_full_gdf))

print("\nIsClearing breakdown:")
print(single_full_gdf["IsClearing"].value_counts(dropna=False))

Total polygons in shapefile: 7

IsClearing breakdown:
IsClearing
u    6
y    1
Name: count, dtype: int64


In [57]:
# Inspect the evaluated polygons for the single pair, including DLL class composition
if single_eval is not None:
    inspect_cols = [
        "_clearing_value",
        "truth",
        "pixel_count",
        "min",
        "mean",
        "max",
        "majority_value",
        "unique_values",
        "prop_clearing_classes",
        "predicted_cleared",
    ]

    for cls in DLL_CLASSES_OF_INTEREST:
        inspect_cols.append(f"class_{cls}_pct")

    inspect_cols = [c for c in inspect_cols if c in single_eval.columns]
    display(single_eval[inspect_cols].head(30))


,_clearing_value,truth,pixel_count,min,mean,max,majority_value,unique_values,prop_clearing_classes,predicted_cleared,class_10_pct,class_34_pct,class_35_pct,class_36_pct,class_37_pct,class_38_pct,class_39_pct
4,Y,1.0,138,10,36.702899,39,39,"10,34,35,36,37,38,39",0.949275,True,5.072464,9.42029,3.623188,2.898551,3.623188,5.072464,70.289855


## 6. Run all matched pairs

This loops over all matched tile/start-date pairs and gives a summary table.


In [58]:

# all_summaries = []
# all_errors = []

# for idx, row in matches.iterrows():
#     try:
#         summary, _, _ = evaluate_match(
#             row,
#             raster_threshold=RASTER_THRESHOLD,
#             min_prop_above_threshold=MIN_PROP_ABOVE_THRESHOLD
#         )
#         all_summaries.append(summary)
#     except Exception as exc:
#         all_errors.append({
#             "match_index": idx,
#             "tile": row.get("tile"),
#             "start_date": row.get("start_date"),
#             "key_qld": row.get("key_qld"),
#             "key_my": row.get("key"),
#             "error": str(exc),
#         })

# results_df = pd.DataFrame(all_summaries)
# errors_df = pd.DataFrame(all_errors)

# print(f"Successful evaluations: {len(results_df)}")
# print(f"Errors: {len(errors_df)}")

# results_df.sort_values(["tile", "start_date"]).head(20)


In [59]:

# # Overall summary
# if not results_df.empty:
#     overall = {
#         "matched_pairs": len(results_df),
#         "validated_cleared_polygons": int(results_df["validated_cleared_polygons"].sum()),
#         "detected_by_my_eds": int(results_df["detected_by_my_eds"].sum()),
#         "missed_by_my_eds": int(results_df["missed_by_my_eds"].sum()),
#     }
#     overall["overall_recall"] = (
#         overall["detected_by_my_eds"] / overall["validated_cleared_polygons"]
#         if overall["validated_cleared_polygons"] else np.nan
#     )

#     display(pd.DataFrame([overall]))

# if not errors_df.empty:
#     display(errors_df.head(20))


## 7. Export results (for later concatenation)

In [60]:
# from pathlib import Path
# import datetime

# # Create output folder
# output_dir = Path("validation_outputs")
# output_dir.mkdir(exist_ok=True)

# # Timestamp to keep runs separate
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# # ---- Export per-polygon evaluation (most detailed) ----
# if 'results_df' in globals() and not results_df.empty:
#     results_path = output_dir / f"eds_validation_summary_{timestamp}.csv"
#     results_df.to_csv(results_path, index=False)
#     print(f"Saved summary results to: {results_path}")

# # ---- Export overall summary ----
# if 'overall' in globals():
#     overall_df = pd.DataFrame([overall])
#     overall_path = output_dir / f"eds_validation_overall_{timestamp}.csv"
#     overall_df.to_csv(overall_path, index=False)
#     print(f"Saved overall summary to: {overall_path}")

# # ---- Export errors (if any) ----
# if 'errors_df' in globals() and not errors_df.empty:
#     errors_path = output_dir / f"eds_validation_errors_{timestamp}.csv"
#     errors_df.to_csv(errors_path, index=False)
#     print(f"Saved errors to: {errors_path}")

# # ---- OPTIONAL: export detailed polygon-level data ----
# if 'single_eval' in globals() and single_eval is not None:
#     detailed_path = output_dir / f"eds_validation_detailed_{timestamp}.csv"
#     single_eval.to_csv(detailed_path, index=False)
#     print(f"Saved detailed polygon results to: {detailed_path}")

In [61]:
all_summaries = []
all_eval_parts = []
all_errors = []

for i, match_row in matches.iterrows():
    try:
        summary, eval_df, full_gdf = evaluate_match(match_row)
        all_summaries.append(summary)

        if eval_df is not None and not eval_df.empty:
            eval_df = eval_df.copy()

            # Put all polygon outputs into a common CRS before concatenation
            if hasattr(eval_df, "to_crs") and getattr(eval_df, "crs", None) is not None:
                eval_df = eval_df.to_crs(COMMON_CRS)

            eval_df["tile"] = match_row["tile"]
            eval_df["start_date"] = match_row["start_date"]
            eval_df["date_group_qld"] = match_row.get("date_group_qld")
            eval_df["date_group_my"] = match_row.get("date_group_my")
            eval_df["key_qld"] = match_row.get("key_qld")
            eval_df["key"] = match_row.get("key")

            all_eval_parts.append(eval_df)

    except Exception as e:
        all_errors.append({
            "match_row_index": i,
            "tile": match_row.get("tile"),
            "start_date": match_row.get("start_date"),
            "date_group_qld": match_row.get("date_group_qld"),
            "date_group_my": match_row.get("date_group_my"),
            "key_qld": match_row.get("key_qld"),
            "key": match_row.get("key"),
            "error": str(e),
        })

results_df = pd.DataFrame(all_summaries)
errors_df = pd.DataFrame(all_errors)

if all_eval_parts:
    all_eval_df = pd.concat(all_eval_parts, ignore_index=True)

    if "geometry" in all_eval_df.columns:
        all_eval_csv_df = all_eval_df.drop(columns=["geometry"]).copy()
    else:
        all_eval_csv_df = all_eval_df.copy()
else:
    all_eval_df = pd.DataFrame()
    all_eval_csv_df = pd.DataFrame()

print(f"Successful evaluations: {len(results_df)}")
print(f"Errors: {len(errors_df)}")

display(results_df.sort_values(["tile", "start_date"]))
if not errors_df.empty:
    display(errors_df)

Successful evaluations: 9
Errors: 0


,tile,start_date,date_group_qld,date_group_my,key_qld,key,product,prediction_method,DLL_CLEARING_CLASSES,MIN_PROP_TARGET_CLASS,...,fn,fp,tn,detected_by_my_eds,missed_by_my_eds,recall,precision,specificity,accuracy,f1
0,p089r078,20250701,d2025070120260117,d2025070120260117,dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/l9olre_p089r078_d2025070120260117_dll_e32756.tif,dll,proportion,"[34, 35, 36, 37, 38, 39]",0.5,...,0,0,0,1,0,1.000000,1.0,NaN,1.000000,1.000000
1,p090r077,20250817,d2025081720260201,d2025081720260201,dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed/lzolre_p090r077_d2025081720260201_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r077/outputs/p090r077_d2025081720260201/l8olre_p090r077_d2025081720260201_dll_e32756.tif,dll,proportion,"[34, 35, 36, 37, 38, 39]",0.5,...,0,0,0,14,0,1.000000,1.0,NaN,1.000000,1.000000
2,p090r079,20251004,d2025100420260201,d2025100420260201,dcceew_202602_run/lzolre_p090r079_d2025100420260201_dlwm6/lzolre_p090r079_d2025100420260201_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r079/outputs/p090r079_d2025100420260201/l8olre_p090r079_d2025100420260201_dll_e32756.tif,dll,proportion,"[34, 35, 36, 37, 38, 39]",0.5,...,1,0,0,9,1,0.900000,1.0,NaN,0.900000,0.947368
3,p090r086,20251105,d2025110520260201,d2025110520260201,dcceew_202602_run/lzolre_p090r086_d2025110520260201_dlwm5/lzolre_p090r086_d2025110520260201_dlwm5_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r086/outputs/p090r086_d2025110520260201/l8olre_p090r086_d2025110520260201_dll_e32755.tif,dll,proportion,"[34, 35, 36, 37, 38, 39]",0.5,...,5,0,0,12,5,0.705882,1.0,NaN,0.705882,0.827586
4,p091r076,20250816,d2025081620260131,d2025081620260131,dcceew_202602_run/lzolre_p091r076_d2025081620260131_dlwm6/lzolre_p091r076_d2025081620260131_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p091r076/outputs/p091r076_d2025081620260131/l9olre_p091r076_d2025081620260131_dll_e32756.tif,dll,proportion,"[34, 35, 36, 37, 38, 39]",0.5,...,0,0,0,7,0,1.000000,1.0,NaN,1.000000,1.000000
5,p091r077,20250917,d2025091720260131,d2025091720260131,dcceew_202602_run/lzolre_p091r077_d2025091720260131_dlwm6/lzolre_p091r077_d2025091720260131_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p091r077/outputs/p091r077_d2025091720260131/l9olre_p091r077_d2025091720260131_dll_e32756.tif,dll,proportion,"[34, 35, 36, 37, 38, 39]",0.5,...,3,0,0,13,3,0.812500,1.0,NaN,0.812500,0.896552
6,p092r076,20251002,d2025100220260122,d2025100220260122,dcceew_202602_run/lzolre_p092r076_d2025100220260122_dlwm5/lzolre_p092r076_d2025100220260122_dlwm5_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p092r076/outputs/p092r076_d2025100220260122/l9olre_p092r076_d2025100220260122_dll_e32755.tif,dll,proportion,"[34, 35, 36, 37, 38, 39]",0.5,...,0,2,0,3,0,1.000000,0.6,0.0,0.600000,0.750000
7,p092r077,20251128,d2025112820260123,d2025112820260123,dcceew_202602_run/lzolre_p092r077_d2025112820260123_dlwm5/lzolre_p092r077_d2025112820260123_dlwm5_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p092r077/outputs/p092r077_d2025112820260123/l9olre_p092r077_d2025112820260123_dll_e32755.tif,dll,proportion,"[34, 35, 36, 37, 38, 39]",0.5,...,0,0,0,3,0,1.000000,1.0,NaN,1.000000,1.000000
8,p092r088,20250731,d2025073120260123,d2025073120260123,dcceew_202602_run/lzolre_p092r088_d2025073120260123_dlwm5/lzolre_p092r088_d2025073120260123_dlwm5_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p092r088/outputs/p092r088_d2025073120260123/l9olre_p092r088_d2025073120260123_dll_e32755.tif,dll,proportion,"[34, 35, 36, 37, 38, 39]",0.5,...,10,0,0,41,10,0.803922,1.0,NaN,0.803922,0.891304


## 6b. Polygon class composition


In [62]:
# Polygon class composition table
# This is the main table for model refinement.
# It shows the percentage of each DLL class inside each QLD validation polygon.

if 'all_eval_csv_df' in globals() and not all_eval_csv_df.empty:
    composition_cols = [
        "tile",
        "start_date",
        "_clearing_value",
        "truth",
        "pixel_count",
        "majority_value",
        "unique_values",
        "prop_clearing_classes",
        "predicted_cleared",
    ]

    for cls in DLL_CLASSES_OF_INTEREST:
        composition_cols.extend([
            f"class_{cls}_count",
            f"class_{cls}_prop",
            f"class_{cls}_pct",
        ])

    composition_cols = [c for c in composition_cols if c in all_eval_csv_df.columns]
    polygon_composition_df = all_eval_csv_df[composition_cols].copy()

    display(polygon_composition_df.head(30))
else:
    polygon_composition_df = pd.DataFrame()
    print("No polygon-level results found yet. Run Section 6 first.")


,tile,start_date,_clearing_value,truth,pixel_count,majority_value,unique_values,prop_clearing_classes,predicted_cleared,class_10_count,...,class_36_pct,class_37_count,class_37_prop,class_37_pct,class_38_count,class_38_prop,class_38_pct,class_39_count,class_39_prop,class_39_pct
0,p089r078,20250701,Y,1.0,138,39,"10,34,35,36,37,38,39",0.949275,True,7,...,2.898551,5,0.036232,3.623188,7,0.050725,5.072464,97,0.702899,70.289855
1,p090r077,20250817,Y,1.0,197,39,"3,10,34,36,37,39",0.857868,True,26,...,1.522843,2,0.010152,1.015228,0,0.000000,0.000000,157,0.796954,79.695431
2,p090r077,20250817,Y,1.0,179,39,"3,34,35,37,38,39",0.988827,True,0,...,0.000000,2,0.011173,1.117318,1,0.005587,0.558659,169,0.944134,94.413408
3,p090r077,20250817,Y,1.0,141,34,"3,10,34,35,36,37,38,39",0.723404,True,36,...,1.418440,1,0.007092,0.709220,3,0.021277,2.127660,14,0.099291,9.929078
4,p090r077,20250817,Y,1.0,137,34,"3,10,34,35,36,39",0.518248,True,62,...,1.459854,0,0.000000,0.000000,0,0.000000,0.000000,5,0.036496,3.649635
5,p090r077,20250817,Y,1.0,127,39,"34,35,36,37,38,39",1.000000,True,0,...,9.448819,10,0.078740,7.874016,8,0.062992,6.299213,68,0.535433,53.543307
6,p090r077,20250817,Y,1.0,123,34,"3,10,34,35,36,37,38,39",0.878049,True,13,...,12.195122,4,0.032520,3.252033,1,0.008130,0.813008,5,0.040650,4.065041
7,p090r077,20250817,Y,1.0,97,39,"3,10,34,35,36,37,38,39",0.814433,True,17,...,5.154639,6,0.061856,6.185567,5,0.051546,5.154639,58,0.597938,59.793814
8,p090r077,20250817,Y,1.0,33,39,"36,37,38,39",1.000000,True,0,...,3.030303,1,0.030303,3.030303,1,0.030303,3.030303,30,0.909091,90.909091
9,p090r077,20250817,Y,1.0,221,39,"3,10,34,35,36,37,38,39",0.769231,True,36,...,8.144796,4,0.018100,1.809955,6,0.027149,2.714932,71,0.321267,32.126697


In [63]:
from pathlib import Path
import datetime

output_dir = Path("validation_outputs")
output_dir.mkdir(exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

method_label = (
    f"{PREDICTION_METHOD}_prop{str(MIN_PROP_TARGET_CLASS).replace('.', 'p')}"
    if PREDICTION_METHOD == "proportion"
    else PREDICTION_METHOD
)

if 'results_df' in globals() and not results_df.empty:
    results_path = output_dir / f"eds_validation_summary_{method_label}_{timestamp}.csv"
    results_df.to_csv(results_path, index=False)
    print(f"Saved summary results: {results_path}")

if 'all_eval_csv_df' in globals() and not all_eval_csv_df.empty:
    eval_path = output_dir / f"eds_validation_polygon_level_{method_label}_{timestamp}.csv"
    all_eval_csv_df.to_csv(eval_path, index=False)
    print(f"Saved polygon-level CSV results: {eval_path}")

if 'polygon_composition_df' in globals() and not polygon_composition_df.empty:
    composition_path = output_dir / f"eds_validation_polygon_composition_{method_label}_{timestamp}.csv"
    polygon_composition_df.to_csv(composition_path, index=False)
    print(f"Saved polygon class composition: {composition_path}")

if 'all_eval_df' in globals() and not all_eval_df.empty and "geometry" in all_eval_df.columns:
    gpkg_path = output_dir / f"eds_validation_polygon_level_{method_label}_{timestamp}.gpkg"
    all_eval_df.to_file(gpkg_path, driver="GPKG")
    print(f"Saved polygon-level spatial results: {gpkg_path}")

if 'errors_df' in globals() and not errors_df.empty:
    errors_path = output_dir / f"eds_validation_errors_{method_label}_{timestamp}.csv"
    errors_df.to_csv(errors_path, index=False)
    print(f"Saved errors: {errors_path}")


Saved summary results: validation_outputs/eds_validation_summary_proportion_prop0p5_20260424_001222.csv
Saved polygon-level CSV results: validation_outputs/eds_validation_polygon_level_proportion_prop0p5_20260424_001222.csv
Saved polygon class composition: validation_outputs/eds_validation_polygon_composition_proportion_prop0p5_20260424_001222.csv
Saved polygon-level spatial results: validation_outputs/eds_validation_polygon_level_proportion_prop0p5_20260424_001222.gpkg


## 8. Notes

This notebook uses the **DLL classified raster**.

DLL class interpretation used here:

- `10` = no change
- `34, 35, 36, 37, 38, 39` = clearing classes at different confidence levels

Prediction method in this notebook:

- `proportion`

The polygon composition table records the percentage of each DLL class within each Queensland validation polygon.


In [64]:
# Optional check: inspect the single-pair class values
if 'single_eval' in globals() and not single_eval.empty:
    display(single_eval[[
        "truth", "_clearing_value", "pixel_count",
        "majority_value", "prop_clearing_classes",
        "unique_values"
    ]].head(30))

,truth,_clearing_value,pixel_count,majority_value,prop_clearing_classes,unique_values
4,1.0,Y,138,39,0.949275,"10,34,35,36,37,38,39"


In [65]:
# Optional overall summary
if 'results_df' in globals() and not results_df.empty:
    overall = {
        "matched_pairs": int(len(results_df)),
        "validated_cleared_polygons": int(results_df["validated_cleared_polygons"].sum()),
        "validated_not_cleared_polygons": int(results_df["validated_not_cleared_polygons"].sum()),
        "tp": int(results_df["tp"].sum()),
        "fn": int(results_df["fn"].sum()),
        "fp": int(results_df["fp"].sum()),
        "tn": int(results_df["tn"].sum()),
    }

    overall["recall"] = overall["tp"] / (overall["tp"] + overall["fn"]) if (overall["tp"] + overall["fn"]) else np.nan
    overall["precision"] = overall["tp"] / (overall["tp"] + overall["fp"]) if (overall["tp"] + overall["fp"]) else np.nan
    overall["specificity"] = overall["tn"] / (overall["tn"] + overall["fp"]) if (overall["tn"] + overall["fp"]) else np.nan
    overall["accuracy"] = (
        (overall["tp"] + overall["tn"]) /
        (overall["tp"] + overall["tn"] + overall["fp"] + overall["fn"])
        if (overall["tp"] + overall["tn"] + overall["fp"] + overall["fn"]) else np.nan
    )
    overall["f1"] = (
        2 * overall["precision"] * overall["recall"] / (overall["precision"] + overall["recall"])
        if pd.notna(overall["precision"]) and pd.notna(overall["recall"]) and (overall["precision"] + overall["recall"]) > 0
        else np.nan
    )

    display(pd.DataFrame([overall]))

,matched_pairs,validated_cleared_polygons,validated_not_cleared_polygons,tp,fn,fp,tn,recall,precision,specificity,accuracy,f1
0,9,122,2,103,19,2,0,0.844262,0.980952,0.0,0.830645,0.907489
